### 해양수산부 국립해양조사원_국가해양관측망 실측 파랑
*from 해양수산부 국립해양조사원*
>국가해양관측망(해양관측부이, 해양과학기지 등)에서 관측하는 파랑(파고, 파주기, 파향) 데이터를 제공 기능

- **제공기관:** 해양수산부 국립해양조사원
- **API 유형:** REST	데이터포맷	JSON+XML
- **End Point:** *https://apis.data.go.kr/1192136/noonWave*
- **Service:**  */GetNoonWaveApiService*  [ 파랑(파고, 파주기, 파향) 데이터 ]

| 항목명 (Field) | 타입 (Type) | 설명 (Description)|
| :--- |:----------| :--- |
| `obsvtrNm` | string    | 관측소명 |
| `lot` | number    | 관측소 경도 |
| `lat` | number    | 관측소 위도 |
| `obsrvnDt` | string    | 관측일시 |
| `wvhgt` | number    | 파고(m) |
| `wvpd` | number    | 파주기(sec) |
| `wvdrct` | number    | 파향(deg) |
| `maxWvhgt` | number    | 최대파고(m) |
| `maxWvpd` | number    | 최대파주기(sec) |

In [1]:
%useLatestDescriptors
%use dataframe

In [4]:
@file:DependsOn("org.xerial:sqlite-jdbc:3.49.1.0")
import java.sql.DriverManager
Class.forName("org.sqlite.JDBC")

class org.sqlite.JDBC

In [2]:
val serviceKeyFilePath = "/Users/unchil/.EnvDataCollector/envDataCollector.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

**[관측소 출처: 오픈API 활용가이드_국가해양관측망 실측 파랑.hwp](https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000003610389&fileDetailSn=1)**

In [ ]:
val filePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/KotlinNotebooks/data/wave_Observatory_list.csv"
val df_observatory = DataFrame.readCsv(filePath, ',')

In [6]:
df_observatory.columnNames()

[code, name]

In [116]:
val tableName_Observatory = "WaveInfoObservatory"
val sql_Observatory = """INSERT INTO $tableName_Observatory (code, name) VALUES (?,?)""".trimIndent()

DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql_Observatory)?.use { pstmt ->
        df_observatory.forEach { row ->
            try {
                pstmt.setString(1, row["code"].toString())
                pstmt.setString(2, row["name"].toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}

In [9]:
val sqlStmt = "SELECT * FROM WaveInfoObservatory"
val df_observatory = DataFrame.readSqlQuery(DriverManager.getConnection(configData.SQLITE_DB.jdbcURL), sqlStmt)

code,name
DT_0039,왕돌초
DT_0042,교본초
IE_0060,이어도
IE_0061,신안가거초
IE_0062,옹진소청초
KG_0021,제주남부
KG_0024,대한해협
KG_0025,남해동부
KG_0028,제주해협
KG_0101,울릉도북동


In [14]:

import kotlinx.datetime.TimeZone
import kotlinx.datetime.format
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlinx.datetime.toLocalDateTime
import kotlin.time.Clock

val endPoint = configData.WAVE_INFO_API.endPoint
val serviceKey = configData.WAVE_INFO_API.apikey
val subPath = configData.WAVE_INFO_API.subPath
val numOfRows = 10
val type = "json"
val min = "5"



* 5분 주기로 수집 ( 마지막 페이지의 결과만 취합)

In [ ]:
val allDataFrames = mutableListOf<DataFrame<*>>()


val now = Clock.System.now()
@OptIn(FormatStringsInDatetimeFormats::class)
val regDate = now
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyyMMdd")})


df_observatory.forEach { row ->
    val obsCode = row["code"].toString()
    val firstUrl = "${endPoint}/${subPath}?serviceKey=${serviceKey}&pageNo=1&numOfRows=1&type=$type&regDate=${regDate}&min=$min&obsCode=$obsCode"
    try{
        val df_first = DataFrame.readJson(firstUrl)
        val totalCnt = df_first["body"]["totalCount"][0].toString().toInt()
        val lastPage = if ((totalCnt % numOfRows) == 0) totalCnt / numOfRows else (totalCnt / numOfRows) + 1
       // println( "obsCode:$obsCode, resultCode:${df_first["header"]["resultCode"][0]}, totalCount:${df_first["body"]["totalCount"][0]}, lastPage:${lastPage}")
        val lastUrl = "${endPoint}/${subPath}?serviceKey=${serviceKey}&pageNo=${lastPage}&numOfRows=${numOfRows}&type=$type&regDate=${regDate}&min=$min&obsCode=$obsCode"
     //   println( lastUrl)
        val df_last = DataFrame.readJson(lastUrl)
        val data = df_last["body"]["items"]["item"].first() as DataFrame<*>
        allDataFrames.add(data)
    }catch(e:Exception){
        println("errMsg:[${e.localizedMessage}],obsCode:[$obsCode], [${firstUrl}]")
    }
}

In [16]:
val concatDf = allDataFrames.concat()
val renamedDf = if (!concatDf.columnNames().contains("wvhgt")) {
    concatDf.rename(
        "ctnWvhgt" to "wvhgt",
        "ctnWvpd" to "wvpd"
    )
} else {
    concatDf
}

val df_cleaned = renamedDf.remove("ctnWvhgt", "ctnWvpd").dropNulls( "wvhgt" )
df_cleaned

obsvtrNm,lot,lat,obsrvnDt,wvdrct,maxWvhgt,maxWvpd,wvhgt,wvpd
이어도,125.182220,32.122768,2026-10-07 16:00,66.989998,1.200000,null,0.700000,4.800000
이어도,125.182220,32.122768,2026-10-07 16:20,65.980003,1.100000,null,0.700000,5.100000
신안가거초,124.592773,33.941940,2026-10-07 16:00,127.610001,0.600000,null,0.400000,3.800000
신안가거초,124.592773,33.941940,2026-10-07 16:20,168.240005,0.600000,null,0.400000,3.800000
제주남부,126.965858,32.090408,2026-10-07 15:00,3.000000,1.850000,7.500000,1.390000,6.000000
제주남부,126.965858,32.090408,2026-10-07 15:30,8.000000,2.170000,5.500000,1.430000,5.700000
제주남부,126.965858,32.090408,2026-10-07 16:00,2.000000,2.050000,6.000000,1.420000,6.100000
대한해협,129.121246,34.918999,2026-10-07 15:00,44.000000,0.970000,5.000000,0.690000,6.000000
대한해협,129.121246,34.918999,2026-10-07 15:30,49.000000,1.060000,5.500000,0.680000,5.800000
대한해협,129.121246,34.918999,2026-10-07 16:00,38.000000,1.050000,7.000000,0.640000,6.000000


In [17]:
val tableName = "WaveInfo"
val sql = """ INSERT OR REPLACE INTO $tableName
 ( obsvtrNm, lot, lat, obsrvnDt, wvhgt, wvpd, wvdrct, maxWvhgt, maxWvpd )
 VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""".trimIndent()

DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql)?.use { pstmt ->
        df_cleaned.forEach { row ->
            try {
                pstmt.setString(1, row["obsvtrNm"]?.toString())
                pstmt.setString(2, row["lot"]?.toString())
                pstmt.setString(3, row["lat"]?.toString())
                pstmt.setString(4, row["obsrvnDt"]?.toString())
                pstmt.setString(5, row["wvhgt"]?.toString())
                pstmt.setString(6, row["wvpd"]?.toString())
                pstmt.setString(7, row["wvdrct"]?.toString())
                pstmt.setString(8, row["maxWvhgt"]?.toString())
                pstmt.setString(9, row["maxWvpd"]?.toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}